# TrOCR: handwritten image -> text

Runs `microsoft/trocr-large-handwritten` (pretrained, no fine-tuning) over every image in a
Drive folder. Unlike `Copy_of_handwrittenmodel_updated.ipynb`, this notebook does **not**
detect a grid / split the page into per-field cells -- each image is preprocessed as a whole
page and split only into text **lines** (TrOCR reads one line at a time), so it suits free-form
handwritten notes rather than forms/tables.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled (Runtime -> Change runtime type -> GPU). TrOCR-large will be slow on CPU.")

In [ ]:
!pip install -q -U "transformers>=4.45" "tokenizers>=0.20" accelerate sentencepiece opencv-python-headless

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    RobertaTokenizer,
)

In [ ]:
DIRECTORY_PATH = "/content/drive/MyDrive/Meenakshi/Dataset/handwriteen/"

image_files = []
for root, _, files in os.walk(DIRECTORY_PATH):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(root, file))

image_files.sort()
print(f"Found {len(image_files)} image files in {DIRECTORY_PATH}")
if len(image_files) > 5:
    print("First 5 image files:", image_files[:5])
else:
    print("Image files:", image_files)

In [ ]:
if image_files:
    current_image_path = image_files[0]
    image = Image.open(current_image_path).convert("RGB")

    print("Image size:", image.size)

    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Original handwritten image: {os.path.basename(current_image_path)}")
    plt.show()
else:
    print("No image files found in the specified directory.")

## Preprocessing

Upscale + contrast-boost + denoise, same idea as the grid notebook: small/faint handwriting
needs the resolution and contrast boost to give the model a chance. This runs on the **whole
page** here (no per-cell cropping).

In [ ]:
def preprocess_handwriting(image, scale=3):
    """
    Enhance a handwriting image for OCR: grayscale, upscale, local contrast boost,
    light denoise. Returns a grayscale numpy array.
    """
    img = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    upscaled = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(upscaled)

    denoised = cv2.fastNlMeansDenoising(
        enhanced,
        None,
        h=5,
        templateWindowSize=7,
        searchWindowSize=21
    )

    return denoised

In [ ]:
PREPROCESS_SCALE = 3
enhanced = preprocess_handwriting(image, scale=PREPROCESS_SCALE)

plt.figure(figsize=(14, 7))
plt.imshow(enhanced, cmap="gray")
plt.axis("off")
plt.title("Enhanced (this is what gets fed to OCR)")
plt.show()

## Split the page into text lines

No grid/cell detection here -- the page is not assumed to be a table. Instead we trim to the
ink bounding box and split the page into individual lines of handwriting using the horizontal
ink profile (rows with almost no ink separate lines). TrOCR reads one line at a time, so each
line is OCR'd separately and the results are joined back together in reading order.

In [ ]:
def ink_mask(gray):
    """Binary mask of dark ink on light paper (Otsu threshold)."""
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return mask


def trim_to_ink(gray, margin_ratio=0.15, min_ink_pixels=10):
    """
    Trim a grayscale image to the bounding box of its ink, then add a white margin.
    Returns None if the image has (almost) no ink.
    """
    if gray.size == 0:
        return None

    mask = ink_mask(gray)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))

    if np.count_nonzero(mask) < min_ink_pixels:
        return None

    ys, xs = np.nonzero(mask)
    y1, y2 = ys.min(), ys.max() + 1
    x1, x2 = xs.min(), xs.max() + 1
    trimmed = gray[y1:y2, x1:x2]

    m = int(min(trimmed.shape) * margin_ratio) + 4
    return cv2.copyMakeBorder(trimmed, m, m, m, m, cv2.BORDER_CONSTANT, value=255)


def split_into_lines(gray, min_line_ratio=0.25, min_gap_ratio=0.4):
    """
    Split a page crop into text-line images using the horizontal ink profile.
    Rows with (almost) no ink separate lines. Returns a list of grayscale arrays,
    top to bottom; returns [gray] when only one line is found.
    """
    mask = ink_mask(gray)
    profile = np.count_nonzero(mask, axis=1)

    if profile.max() == 0:
        return [gray]

    has_ink = profile > profile.max() * 0.05

    runs = []
    start = None
    for y, ink in enumerate(has_ink):
        if ink and start is None:
            start = y
        elif not ink and start is not None:
            runs.append([start, y])
            start = None
    if start is not None:
        runs.append([start, len(has_ink)])

    if not runs:
        return [gray]

    # merge runs separated by small gaps (dots, descenders, ascenders)
    median_height = np.median([b - a for a, b in runs])
    min_gap = max(2, int(median_height * min_gap_ratio))
    merged = [runs[0]]
    for run in runs[1:]:
        if run[0] - merged[-1][1] < min_gap:
            merged[-1][1] = run[1]
        else:
            merged.append(run)

    # drop runs much shorter than the tallest one (noise, stray marks)
    tallest = max(b - a for a, b in merged)
    merged = [r for r in merged if (r[1] - r[0]) >= tallest * min_line_ratio]

    if len(merged) <= 1:
        return [gray]

    pad = max(2, min_gap // 2)
    return [gray[max(0, a - pad):min(gray.shape[0], b + pad)] for a, b in merged]


def prepare_page_for_trocr(enhanced):
    """
    Full page preparation: trim to ink, split into lines, trim each line.
    Returns a list of RGB PIL line images (empty list if no ink found).
    """
    trimmed = trim_to_ink(enhanced)
    if trimmed is None:
        return []

    line_images = []
    for line in split_into_lines(trimmed):
        line = trim_to_ink(line)
        if line is not None:
            line_images.append(Image.fromarray(line).convert("RGB"))
    return line_images

In [ ]:
page_lines = prepare_page_for_trocr(enhanced)
print(f"Detected {len(page_lines)} line(s) in {os.path.basename(current_image_path)}")

n = len(page_lines)
if n:
    fig, axes = plt.subplots(n, 1, figsize=(10, max(n, 1) * 1.5))
    if n == 1:
        axes = [axes]
    for ax, line_img in zip(axes, page_lines):
        ax.imshow(line_img, cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Load TrOCR (`microsoft/trocr-large-handwritten`)

Forces the slow (BPE) `RobertaTokenizer` instead of letting `AutoTokenizer` try to build a
fast tokenizer -- that conversion path can raise "Couldn't instantiate the backend
tokenizer" on this repo even with `sentencepiece` installed. The slow tokenizer only needs
`vocab.json`/`merges.txt`, which this repo has, so it loads directly with no conversion step.

If loading still raises that error after installing packages above, restart the runtime
(Runtime -> Restart session) and re-run all cells from the top.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

TROCR_MODEL_ID = "microsoft/trocr-large-handwritten"

print("Loading TrOCR...")

image_processor = AutoImageProcessor.from_pretrained(TROCR_MODEL_ID)
tokenizer = RobertaTokenizer.from_pretrained(TROCR_MODEL_ID, use_fast=False)
trocr_processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

trocr_model = VisionEncoderDecoderModel.from_pretrained(
    TROCR_MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)
trocr_model.eval()

print("TrOCR loaded on", device)

In [ ]:
def trocr_ocr_batch(line_images, batch_size=16, num_beams=4, max_new_tokens=64):
    """
    Run TrOCR on a list of single-line PIL images.
    Returns (texts, confidences) -- confidences are per-line probabilities derived
    from the beam search's length-normalized sequence log-probability
    (`sequences_scores`), a simple proxy for how sure the model was.
    """
    texts = []
    confidences = []
    for i in range(0, len(line_images), batch_size):
        batch = line_images[i:i + batch_size]
        pixel_values = trocr_processor(images=batch, return_tensors="pt").pixel_values
        pixel_values = pixel_values.to(device, dtype=trocr_model.dtype)

        with torch.inference_mode():
            outputs = trocr_model.generate(
                pixel_values,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True,
            )

        texts.extend(
            t.strip() for t in trocr_processor.batch_decode(outputs.sequences, skip_special_tokens=True)
        )
        confidences.extend(
            torch.exp(outputs.sequences_scores).tolist()
        )
    return texts, confidences


def trocr_read_image(image, scale=PREPROCESS_SCALE):
    """
    Preprocess + line-split + OCR one image (no cell/grid detection).
    Returns (full_text, line_images, line_texts, line_confidences, confidence).
    `confidence` is the mean of the per-line confidences (page-level proxy).
    """
    enhanced = preprocess_handwriting(image, scale=scale)
    line_images = prepare_page_for_trocr(enhanced)
    if line_images:
        line_texts, line_confidences = trocr_ocr_batch(line_images)
    else:
        line_texts, line_confidences = [], []
    full_text = "\n".join(line_texts)
    confidence = float(np.mean(line_confidences)) if line_confidences else float("nan")
    return full_text, line_images, line_texts, line_confidences, confidence

## Try it on one image

In [ ]:
preview_image = Image.open(image_files[0]).convert("RGB")
preview_text, preview_lines, preview_line_texts, preview_line_confidences, preview_confidence = trocr_read_image(preview_image)

print(f"{os.path.basename(image_files[0])}: {len(preview_lines)} line(s)\n")
print(f"Confidence: {preview_confidence:.4f}")
print("OCR text:")
print(preview_text)

## Run TrOCR on every image and save results

In [ ]:
trocr_results = {}

for img_path in image_files:
    print(f"Processing image: {os.path.basename(img_path)}")
    try:
        image = Image.open(img_path).convert("RGB")
        full_text, line_images, line_texts, line_confidences, confidence = trocr_read_image(image)
        trocr_results[img_path] = {
            "text": full_text,
            "lines": line_texts,
            "line_confidences": line_confidences,
            "confidence": confidence,
        }
        print(f"-> {len(line_texts)} line(s) read (confidence: {confidence:.4f})")
    except Exception as e:
        print(f"Error processing {os.path.basename(img_path)}: {e}")

processed_data = [
    {"image_name": os.path.basename(p), "ocr_text": r["text"], "confidence": r["confidence"]}
    for p, r in trocr_results.items()
]
final_df = pd.DataFrame(processed_data)
print(f"\nProcessed {len(final_df)} image(s).")

OUTPUT_DIR = "/content/drive/MyDrive/Meenakshi/Dataset/ocr_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "trocr_results.csv")
json_path = os.path.join(OUTPUT_DIR, "trocr_results.json")

final_df.to_csv(csv_path, index=False, encoding="utf-8")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {os.path.basename(p): r for p, r in trocr_results.items()},
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)

In [ ]:
final_df